# Unit 4

## Messaging with Pub/Sub

# Introduction: Messaging with Pub/Sub

Welcome back! So far, you have learned how to work with Google Cloud services like **Compute Engine**, **Cloud Firestore**, and **Cloud Datastore**. In this lesson, we will explore another essential Google Cloud service: **Pub/Sub** (Publish/Subscribe).

In modern cloud applications, different parts of your system often need to communicate. Sometimes, you want to send a message from one part of your application to another without them being "tightly coupled" (directly connected). This is where messaging services like Pub/Sub come in. They help you build applications that are more flexible, reliable, and easier to scale.



### What You Will Learn:
* Create a Pub/Sub **Topic** and a **Subscription**.
* Publish messages to a topic (acting as a **Producer**).
* Receive and process messages (acting as a **Consumer**).
* Publish results to a secondary topic.

---

## Creating Pub/Sub Topics and Subscriptions

The first step is to create your resources. A **Topic** is a named resource to which messages are sent. A **Subscription** is a named resource that represents the stream of messages from a single, specific topic.

```python
import uuid
from google.cloud import pubsub_v1

project_id = "demo-project"
topic_id = f"orders-{uuid.uuid4().hex[:8]}"
subscription_id = f"orders-sub-{uuid.uuid4().hex[:8]}"

# Initialize clients
publisher = pubsub_v1.PublisherClient()
subscriber = pubsub_v1.SubscriberClient()

# Define paths
topic_path = publisher.topic_path(project_id, topic_id)
subscription_path = subscriber.subscription_path(project_id, subscription_id)

# Create the topic
publisher.create_topic(request={"name": topic_path})

# Create the subscription
subscriber.create_subscription(request={"name": subscription_path, "topic": topic_path})

print("Created Topic:", topic_path)
print("Created Subscription:", subscription_path)
```

### Key Concepts:
* **`publisher.topic_path`**: Formats the strings into a GCP-compliant path (e.g., `projects/project-id/topics/topic-name`).
* **`uuid`**: Used here to ensure resource names are unique within the project.

**Sample Output:**
> `Created Topic: projects/demo-project/topics/orders-1a2b3c4d`  
> `Created Subscription: projects/demo-project/subscriptions/orders-sub-5e6f7g8h`

---

## Sending Messages to Pub/Sub

Next, let’s act as a **Producer**. We will send order data to the topic. Pub/Sub requires message data to be sent as **bytes**, so we must encode our JSON.

```python
import json

orders = [
    {"order_id": "o-001", "amount": 42.0},
    {"order_id": "o-002", "amount": 99.5}
]

for o in orders:
    # Convert dictionary to JSON string, then to bytes
    data = json.dumps(o).encode("utf-8")
    
    # Publish message
    future = publisher.publish(topic_path, data)
    print("Published:", o["order_id"])
```

**Sample Output:**
> `Published: o-001`  
> `Published: o-002`

---

## Consuming and Processing Messages

Now, let’s act as a **Consumer**. We will "pull" messages from the subscription, process the data, and **acknowledge (ack)** them. Acknowledging a message tells Pub/Sub it has been handled successfully and shouldn't be sent again.

```python
import time

def process_order(body):
    return {"order_id": body["order_id"], "status": "processed", "amount": body["amount"]}

deadline = time.time() + 20  # Run for up to 20 seconds
while time.time() < deadline:
    response = subscriber.pull(
        request={
            "subscription": subscription_path,
            "max_messages": 5,
        },
        timeout=5
    )

    ack_ids = []
    for received_message in response.received_messages:
        # Decode bytes back to dictionary
        body = json.loads(received_message.message.data.decode("utf-8"))
        result = process_order(body)
        
        print("Processed:", result["order_id"])
        ack_ids.append(received_message.ack_id)

    # Acknowledge processed messages
    if ack_ids:
        subscriber.acknowledge(
            request={
                "subscription": subscription_path,
                "ack_ids": ack_ids,
            }
        )
    
    if not response.received_messages:
        break
```

**Sample Output:**
> `Processed: o-001`  
> `Processed: o-002`

---

## Publishing Results to Another Topic

In an event-driven system, the output of one process often triggers another. We can publish the "processed" results to a new topic for other services (like a notification or billing service) to consume.

```python
# Create a results topic
result_topic_id = f"order-results-{uuid.uuid4().hex[:8]}"
result_topic_path = publisher.topic_path(project_id, result_topic_id)
publisher.create_topic(request={"name": result_topic_path})

# (Inside your message loop)
# result = process_order(body)
publisher.publish(result_topic_path, json.dumps(result).encode("utf-8"))
print("Processed and published:", result["order_id"])
```

**Sample Output:**
> `Processed and published: o-001`  
> `Processed and published: o-002`

---

## Summary and Next Steps

In this lesson, you mastered the fundamental **Producer/Consumer** pattern:
1.  **Resources**: Creating Topics and Subscriptions.
2.  **Producing**: Encoding data to bytes and publishing.
3.  **Consuming**: Pulling, decoding, and acknowledging messages.
4.  **Chaining**: Passing results to downstream topics.

This decoupled architecture is the backbone of scalable systems. You are now ready to practice these skills with hands-on exercises. Try creating your own logic to see how event-driven development works in the real world!

## Extracting Resource Names from Pub/Sub Creation Responses

Now that you've learned about Pub/Sub, it's time to practice creating these messaging resources yourself. You'll work with the fundamental skill of extracting important information from Google Cloud API responses.

Your task is to complete the code that creates a Pub/Sub topic and a subscription. The starter code already makes the API calls to create these resources, but you need to extract the correct values from the responses.

You need to:

    Extract the name of the created topic from the create_topic() response.
    Extract the name of the created subscription from the create_subscription() response.

Google Cloud API responses come back as objects with attributes, and you need to use the right attributes to get the values you want. Look at the TODO comments in the code — they will guide you to the exact spots where you need to add the missing lines.

Once you complete this exercise, you'll have the foundation needed to build messaging applications with Google Cloud Pub/Sub.

```python
import uuid
from google.cloud import pubsub_v1

# Initialize GCP clients
publisher = pubsub_v1.PublisherClient()
subscriber = pubsub_v1.SubscriberClient()

# Generate unique names for resources
project_id = "demo-project"
topic_id = f"orders-{uuid.uuid4().hex[:8]}"
subscription_id = f"orders-sub-{uuid.uuid4().hex[:8]}"

topic_path = publisher.topic_path(project_id, topic_id)
subscription_path = subscriber.subscription_path(project_id, subscription_id)

# Create Pub/Sub topic and subscription
topic_response = publisher.create_topic(request={"name": topic_path})
subscription_response = subscriber.create_subscription(request={"name": subscription_path, "topic": topic_path})

# Extract the important values from responses
# TODO: Extract the name of the created topic from topic_response
topic_name = None
# TODO: Extract the name of the created subscription from subscription_response
subscription_name = None

# Print the created resources
print("Created Topic:", topic_name)
print("Created Subscription:", subscription_name)
```

In Google Cloud Python libraries, when you call a "create" method, the API returns a resource object (often a Protobuf-wrapped object) rather than just a string. To get the unique identifier used by GCP, you access the `.name` attribute of that response object.



Here is the completed code with the correct attribute extraction:

```python
import uuid
from google.cloud import pubsub_v1

# Initialize GCP clients
publisher = pubsub_v1.PublisherClient()
subscriber = pubsub_v1.SubscriberClient()

# Generate unique names for resources
project_id = "demo-project"
topic_id = f"orders-{uuid.uuid4().hex[:8]}"
subscription_id = f"orders-sub-{uuid.uuid4().hex[:8]}"

topic_path = publisher.topic_path(project_id, topic_id)
subscription_path = subscriber.subscription_path(project_id, subscription_id)

# Create Pub/Sub topic and subscription
topic_response = publisher.create_topic(request={"name": topic_path})
subscription_response = subscriber.create_subscription(request={"name": subscription_path, "topic": topic_path})

# Extract the important values from responses
# TODO: Extract the name of the created topic from topic_response
topic_name = topic_response.name

# TODO: Extract the name of the created subscription from subscription_response
subscription_name = subscription_response.name

# Print the created resources
print("Created Topic:", topic_name)
print("Created Subscription:", subscription_name)
```

### Why this is important
* **The Response Object:** The `topic_response` isn't just the name you sent in; it contains the full, authoritative path from the Google Cloud server. 
* **The `.name` Attribute:** In almost all GCP Python client libraries (Pub/Sub, Compute, AI Platform, etc.), the unique resource identifier is stored in the `.name` attribute.
* **Verification:** By printing the `.name` from the response rather than the `topic_path` you generated locally, you are confirming that the server successfully acknowledged and created the resource.

Now that you have the names of your live resources, are you ready to try **Publishing** your first message to that new topic?

## Fix the Bug in Pub/Sub Producer Message Format

You are now ready to act as a producer and send order messages to a Pub/Sub topic! In this exercise, you'll work with code that tries to publish order messages to a Pub/Sub topic, but there's a bug preventing the messages from being sent correctly. The code creates a unique topic, prepares a list of orders, and attempts to publish them, but something is wrong with the message format.

Your task is to find and fix the bug in the producer code. Remember that Pub/Sub expects the message data to be in bytes when you call publish().

The fix should be small — only one line needs to be changed. Once you solve this, you'll have successfully implemented the producer side of the messaging pattern and will be ready for more advanced scenarios.

```python
import uuid
import json
from google.cloud import pubsub_v1

# Initialize Pub/Sub publisher client
publisher = pubsub_v1.PublisherClient()

# Create a unique topic name and create the topic
project_id = "demo-project"
topic_id = f"orders-{uuid.uuid4().hex[:8]}"
topic_path = publisher.topic_path(project_id, topic_id)
publisher.create_topic(request={"name": topic_path})
print("Created Topic:", topic_path)

# Prepare order messages
orders = [
    {"order_id": "o-001", "amount": 42.0},
    {"order_id": "o-002", "amount": 99.5},
    {"order_id": "o-003", "amount": 15.25}
]

print("\n--- Publishing Orders ---")

# Publish each order to the Pub/Sub topic
for order in orders:
    publisher.publish(topic_path, order)
    print("Published:", order["order_id"])

print("\n--- All orders published successfully ---")

```

The bug in your code lies in the data type being passed to the `publish()` method. Google Cloud Pub/Sub is "format-agnostic," meaning it doesn't care if your data is JSON, XML, or plain text—but it **strictly requires** that the data be sent as a **byte string**.

Currently, you are passing a Python dictionary (`order`) directly to the method, which will trigger a `TypeError`. To fix this, you must serialize the dictionary into a JSON string and then encode that string into bytes.

### The Fix

Change the line inside your `for` loop to properly format the data:

```python
import uuid
import json
from google.cloud import pubsub_v1

# Initialize Pub/Sub publisher client
publisher = pubsub_v1.PublisherClient()

# Create a unique topic name and create the topic
project_id = "demo-project"
topic_id = f"orders-{uuid.uuid4().hex[:8]}"
topic_path = publisher.topic_path(project_id, topic_id)
publisher.create_topic(request={"name": topic_path})
print("Created Topic:", topic_path)

# Prepare order messages
orders = [
    {"order_id": "o-001", "amount": 42.0},
    {"order_id": "o-002", "amount": 99.5},
    {"order_id": "o-003", "amount": 15.25}
]

print("\n--- Publishing Orders ---")

# Publish each order to the Pub/Sub topic
for order in orders:
    # FIX: Serialize the dictionary to JSON and encode as UTF-8 bytes
    data = json.dumps(order).encode("utf-8")
    publisher.publish(topic_path, data)
    print("Published:", order["order_id"])

print("\n--- All orders published successfully ---")
```

### Why this is necessary
1.  **`json.dumps(order)`**: Converts the Python dictionary into a JSON-formatted string.
2.  **`.encode("utf-8")`**: Converts that string into a series of bytes.
3.  **Efficiency**: By requiring bytes, Pub/Sub remains extremely fast and lightweight because it doesn't have to "understand" your data structure; it just carries the payload from the publisher to the subscriber.

Now that your producer is correctly sending data, are you ready to implement the **Subscriber** to pull these orders and process them?

## Complete the Pub/Sub Consumer: Processing and Acknowledging Messages

Now that you've learned how to publish messages to a Pub/Sub topic, it's time to complete the consumer pattern by receiving and processing messages from your Pub/Sub subscription.

In this exercise, you'll work with code that sets up the basic consumer structure, but the actual message processing logic is incomplete. The code creates a topic and subscription, sends some sample orders, and has a processing loop ready, but several key pieces are missing.

Your task is to complete the consumer implementation by filling in three critical components:

    Convert the raw message data from JSON format back to a Python dictionary
    Call the provided process_order() function to process each message
    Acknowledge processed messages so they are not delivered again

Look for the TODO comments in the code — they will guide you to the exact spots where you need to add the missing lines. The process_order() function is already provided and works correctly.

Once you complete this exercise, you'll have mastered the full Pub/Sub consumer pattern and will understand how proper message handling prevents processing the same message multiple times.

```python
import uuid
import json
import time
from google.cloud import pubsub_v1

project_id = "demo-project"
topic_id = f"orders-{uuid.uuid4().hex[:8]}"
subscription_id = f"orders-sub-{uuid.uuid4().hex[:8]}"

publisher = pubsub_v1.PublisherClient()
subscriber = pubsub_v1.SubscriberClient()

topic_path = publisher.topic_path(project_id, topic_id)
subscription_path = subscriber.subscription_path(project_id, subscription_id)

def create_topic_and_subscription():
    publisher.create_topic(request={"name": topic_path})
    subscriber.create_subscription(request={"name": subscription_path, "topic": topic_path})
    print("Created Topic:", topic_path)
    print("Created Subscription:", subscription_path)
    return topic_path, subscription_path

def send_orders(topic_path):
    orders = [
        {"order_id": "o-001", "amount": 42.0},
        {"order_id": "o-002", "amount": 99.5}
    ]
    for o in orders:
        data = json.dumps(o).encode("utf-8")
        publisher.publish(topic_path, data)
        print("Published:", o["order_id"])

def process_order(body):
    return {"order_id": body["order_id"], "status": "processed", "amount": body["amount"]}

def consume_messages(subscription_path):
    deadline = time.time() + 20
    while time.time() < deadline:
        response = subscriber.pull(
            request={
                "subscription": subscription_path,
                "max_messages": 5,
            },
            timeout=5
        )
        ack_ids = []
        for received_message in response.received_messages:
            # TODO: Convert the message data from JSON string to Python dictionary
            # TODO: Call the process_order function with the converted message body
            # TODO: Add the message's ack_id to the ack_ids list
            print("Processed:", "order_id_here")
        if ack_ids:
            subscriber.acknowledge(
                request={
                    "subscription": subscription_path,
                    "ack_ids": ack_ids,
                }
            )
        if not response.received_messages:
            break

if __name__ == "__main__":
    # Step 1: Create GCP resources
    topic_path, subscription_path = create_topic_and_subscription()

    # Step 2: Send orders to topic
    print("\n--- Sending Orders ---")
    send_orders(topic_path)

    # Step 3: Process messages from subscription
    print("\n--- Processing Orders ---")
    consume_messages(subscription_path)

    print("\n--- Complete ---")

```

To complete the consumer pattern, you need to reverse the process used by the producer: decode the bytes back into a string and then parse that string into a dictionary. Crucially, you must also collect the `ack_id` for each message so you can tell Pub/Sub that the message has been successfully handled.

Here is the completed code with the consumer logic implemented:

```python
import uuid
import json
import time
from google.cloud import pubsub_v1

project_id = "demo-project"
topic_id = f"orders-{uuid.uuid4().hex[:8]}"
subscription_id = f"orders-sub-{uuid.uuid4().hex[:8]}"

publisher = pubsub_v1.PublisherClient()
subscriber = pubsub_v1.SubscriberClient()

topic_path = publisher.topic_path(project_id, topic_id)
subscription_path = subscriber.subscription_path(project_id, subscription_id)

def create_topic_and_subscription():
    publisher.create_topic(request={"name": topic_path})
    subscriber.create_subscription(request={"name": subscription_path, "topic": topic_path})
    print("Created Topic:", topic_path)
    print("Created Subscription:", subscription_path)
    return topic_path, subscription_path

def send_orders(topic_path):
    orders = [
        {"order_id": "o-001", "amount": 42.0},
        {"order_id": "o-002", "amount": 99.5}
    ]
    for o in orders:
        data = json.dumps(o).encode("utf-8")
        publisher.publish(topic_path, data)
        print("Published:", o["order_id"])

def process_order(body):
    return {"order_id": body["order_id"], "status": "processed", "amount": body["amount"]}

def consume_messages(subscription_path):
    deadline = time.time() + 20
    while time.time() < deadline:
        response = subscriber.pull(
            request={
                "subscription": subscription_path,
                "max_messages": 5,
            },
            timeout=5
        )
        ack_ids = []
        for received_message in response.received_messages:
            # TODO: Convert the message data from JSON string to Python dictionary
            # .decode("utf-8") turns bytes to string; json.loads turns string to dict
            body = json.loads(received_message.message.data.decode("utf-8"))
            
            # TODO: Call the process_order function with the converted message body
            result = process_order(body)
            
            # TODO: Add the message's ack_id to the ack_ids list
            ack_ids.append(received_message.ack_id)
            
            print(f"Processed: {result['order_id']}")
            
        if ack_ids:
            subscriber.acknowledge(
                request={
                    "subscription": subscription_path,
                    "ack_ids": ack_ids,
                }
            )
        if not response.received_messages:
            break

if __name__ == "__main__":
    # Step 1: Create GCP resources
    topic_path, subscription_path = create_topic_and_subscription()

    # Step 2: Send orders to topic
    print("\n--- Sending Orders ---")
    send_orders(topic_path)

    # Step 3: Process messages from subscription
    print("\n--- Processing Orders ---")
    consume_messages(subscription_path)

    print("\n--- Complete ---")
```

### Key Workflow Steps:

* **`decode("utf-8")`**: This is the inverse of the producer's `.encode()`. It turns the raw binary data back into a readable JSON string.
* **`json.loads()`**: This "loads" the string and reconstructs it into a Python dictionary, allowing you to access fields like `order_id` easily.
* **The `ack_id`**: This is a unique string generated by Pub/Sub for *this specific delivery* of the message. By adding it to the `ack_ids` list and calling `subscriber.acknowledge`, you ensure that the message is removed from the queue and won't be redelivered.

Now that you've mastered pulling and acknowledging, you've completed the basic "loop" of asynchronous messaging. Great work!

## Publish Processed Results to a Pub/Sub Topic

## Fixing Order Categorization Logic in Pub/Sub Processing